In [1]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)
n_samples = 5000

# Generating features
data = {
    'customer_age': np.random.randint(18, 70, size=n_samples),
    'purchase_amount': np.round(np.random.uniform(10, 500, size=n_samples), 2),
    'product_category': np.random.choice(['Clothing', 'Electronics', 'Home', 'Beauty'], size=n_samples, p=[0.4, 0.2, 0.2, 0.2]),
    'discount_applied': np.random.choice([0, 1], size=n_samples, p=[0.6, 0.4]),
    'customer_rating': np.random.randint(1, 6, size=n_samples),
    'previous_returns': np.random.poisson(lam=0.8, size=n_samples)
}

df = pd.DataFrame(data)

# Logistic relation to construct realistic target variable (Returned: 0 or 1)
log_odds = (
    0.05 * (df['product_category'] == 'Clothing') * df['purchase_amount'] 
    + 0.4 * df['previous_returns'] 
    - 0.3 * df['customer_rating'] 
    - 1.5
)
probabilities = 1 / (1 + np.exp(-log_odds))
df['returned'] = np.random.binomial(1, probabilities)

print(f"Dataset generated with shape: {df.shape}")
print(f"Return Rate (Class Imbalance Check):\n{df['returned'].value_counts(normalize=True)}")

Dataset generated with shape: (5000, 7)
Return Rate (Class Imbalance Check):
returned
0    0.5602
1    0.4398
Name: proportion, dtype: float64


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Split Features and Target
X = df.drop('returned', axis=1)
y = df['returned']

# Identify numerical and categorical columns
num_features = ['customer_age', 'purchase_amount', 'customer_rating', 'previous_returns']
cat_features = ['product_category', 'discount_applied']

# Define preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ])

# Split Train/Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Data split successfully.")

Data split successfully.


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Construct an end-to-end ML Pipeline handling class imbalance via SMOTE
model_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

# Train the model
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       560
           1       0.98      0.84      0.90       440

    accuracy                           0.92      1000
   macro avg       0.93      0.91      0.92      1000
weighted avg       0.92      0.92      0.92      1000

ROC-AUC Score: 0.9376


In [4]:
# Create a simulated testing dataframe to calculate business impact
test_analysis = X_test.copy()
test_analysis['true_return'] = y_test
test_analysis['predicted_return'] = y_pred

# Assume an average cost of handling a return (reverse logistics) is $15
RETURN_LOGISTICS_COST = 15.00

# Quantify total financial risk predicted by your model
total_risk_detected = (test_analysis['predicted_return'] * RETURN_LOGISTICS_COST).sum()
print(f"🚀 Business Impact Summary:")
print(f"Total reverse logistics financial risk flag identified by AI: ${total_risk_detected:,.2f}")
print("By targeting these high-risk orders with localized optimizations, company can save significant overhead.")

🚀 Business Impact Summary:
Total reverse logistics financial risk flag identified by AI: $5,655.00
By targeting these high-risk orders with localized optimizations, company can save significant overhead.


In [5]:
import joblib

# Save the trained ML pipeline (includes preprocessing + SMOTE + Random Forest)
joblib.dump(model_pipeline, 'return_prediction_model.pkl')
print("✅ Model successfully serialized and saved as 'return_prediction_model.pkl'")

✅ Model successfully serialized and saved as 'return_prediction_model.pkl'


In [6]:
%%writefile app.py
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

# 1. Initialize the FastAPI application
app = FastAPI(
    title="E-Commerce Return Prediction API", 
    description="API to predict the risk and financial cost of product returns.",
    version="1.0"
)

# 2. Load the pre-trained pipeline
model = joblib.load('return_prediction_model.pkl')

# 3. Define the structural schema for incoming checkout requests
class OrderInput(BaseModel):
    customer_age: int
    purchase_amount: float
    product_category: str
    discount_applied: int
    customer_rating: int
    previous_returns: int

# 4. Create the API endpoint
@app.post("/predict")
def predict_return(order: OrderInput):
    # Convert incoming JSON payload into a Pandas DataFrame format expected by the model
    input_data = pd.DataFrame([order.model_dump()])
    
    # Run predictions
    prediction = int(model.predict(input_data)[0])
    probability = float(model.predict_proba(input_data)[0][1])
    
    # Financial Impact calculation ($15 default reverse logistics cost)
    logistics_cost_risk = 15.00 if prediction == 1 else 0.00
    
    return {
        "return_predicted": prediction,
        "return_probability": round(probability, 4),
        "estimated_financial_risk_usd": logistics_cost_risk,
        "recommendation": "Flag for logistics optimization" if prediction == 1 else "Standard Processing"
    }

Writing app.py


In [10]:
%pip install fastapi uvicorn


   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   ---------------------------------------- 3/3 [fastapi]

Note: you may need to restart the kernel to use updated packages.


In [11]:
import threading
import uvicorn

# Define function to run uvicorn
def run_server():
    uvicorn.run("app:app", host="127.0.0.1", port=8000, log_level="info")

# Start server in a non-blocking background thread so your notebook stays responsive
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 Live FastAPI Server running at: http://127.0.0.1:8000")
print("📖 Interactive API documentation available at: http://127.0.0.1:8000/docs")

🚀 Live FastAPI Server running at: http://127.0.0.1:8000
📖 Interactive API documentation available at: http://127.0.0.1:8000/docs


INFO:     Started server process [22044]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:59393 - "POST /predict HTTP/1.1" 200 OK


In [12]:
import requests

# Define a simulated customer profile testing the API
test_checkout_payload = {
    "customer_age": 34,
    "purchase_amount": 250.50,
    "product_category": "Clothing",
    "discount_applied": 1,
    "customer_rating": 2,
    "previous_returns": 4
}

# Send a POST request to your local API endpoint
response = requests.post("http://127.0.0.1:8000/predict", json=test_checkout_payload)

print("--- API Server JSON Response ---")
import json
print(json.dumps(response.json(), indent=4))

--- API Server JSON Response ---
{
    "return_predicted": 1,
    "return_probability": 0.9859,
    "estimated_financial_risk_usd": 15.0,
    "recommendation": "Flag for logistics optimization"
}
